**House Prices - Model Experiments**

In [55]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/house-prices-advanced-regression-techniques/sample_submission.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/data_description.txt
/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv


*Data Loading* 

In [56]:
train = pd.read_csv("/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv")
test = pd.read_csv("/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv")

print(train.shape)
print(test.shape)

print(train.head())

(1460, 81)
(1459, 80)
   Id  MSSubClass MSZoning  LotFrontage  LotArea Street Alley LotShape  \
0   1          60       RL         65.0     8450   Pave   NaN      Reg   
1   2          20       RL         80.0     9600   Pave   NaN      Reg   
2   3          60       RL         68.0    11250   Pave   NaN      IR1   
3   4          70       RL         60.0     9550   Pave   NaN      IR1   
4   5          60       RL         84.0    14260   Pave   NaN      IR1   

  LandContour Utilities  ... PoolArea PoolQC Fence MiscFeature MiscVal MoSold  \
0         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      2   
1         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      5   
2         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      9   
3         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      2   
4         Lvl    AllPub  ...        0    NaN   NaN         NaN       0     12   

  YrSold  SaleType  SaleCondition  SalePrice  

In [57]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   Alley          91 non-null     object 
 7   LotShape       1460 non-null   object 
 8   LandContour    1460 non-null   object 
 9   Utilities      1460 non-null   object 
 10  LotConfig      1460 non-null   object 
 11  LandSlope      1460 non-null   object 
 12  Neighborhood   1460 non-null   object 
 13  Condition1     1460 non-null   object 
 14  Condition2     1460 non-null   object 
 15  BldgType       1460 non-null   object 
 16  HouseStyle     1460 non-null   object 
 17  OverallQual    1460 non-null   int64  
 18  OverallC

In [58]:
train.isnull().sum().sort_values(ascending=False).head(20)

PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
LotFrontage      259
GarageQual        81
GarageFinish      81
GarageType        81
GarageYrBlt       81
GarageCond        81
BsmtFinType2      38
BsmtExposure      38
BsmtCond          37
BsmtQual          37
BsmtFinType1      37
MasVnrArea         8
Electrical         1
Condition2         0
dtype: int64

*Data Cleaning*

In [59]:
y = train["SalePrice"]
X = train.drop("SalePrice", axis=1)

In [60]:
test_ids = X["Id"]
X = X.drop("Id", axis=1)

In [61]:
num_cols = X.select_dtypes(include=["int64", "float64"]).columns
X[num_cols] = X[num_cols].fillna(X[num_cols].median())
test[num_cols] = test[num_cols].fillna(test[num_cols].median())

In [62]:
cat_cols = X.select_dtypes(include=["object"]).columns
X[cat_cols] = X[cat_cols].fillna("None")
test[cat_cols] = test[cat_cols].fillna("None")

*Feature Engineering*

In [63]:
X["TotalSF"] = X["TotalBsmtSF"] + X["1stFlrSF"] + X["2ndFlrSF"]
test["TotalSF"] = test["TotalBsmtSF"] + test["1stFlrSF"] + test["2ndFlrSF"]

X["Age"] = X["YrSold"] - X["YearBuilt"]
test["Age"] = test["YrSold"] - test["YearBuilt"]

*Encoding*

In [64]:
X = pd.get_dummies(X)
test = pd.get_dummies(test)

X, test = X.align(test, join='left', axis=1, fill_value=0)

*Feature Selection*

In [65]:
print("Number of features:", X.shape[1])

Number of features: 305


*Test Train Split*

In [66]:
from sklearn.model_selection import train_test_split

y = np.log1p(y)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

*Mlflow Setup*

In [67]:
!pip install mlflow dagshub

In [68]:
import mlflow
import dagshub

dagshub.init(repo_owner='slosa23', repo_name='ML-Assignment1', mlflow=True)

Initialized MLflow to track repo "slosa23/ML-Assignment1"

Repository slosa23/ML-Assignment1 initialized!

*Model Training - Linear Regression*

In [69]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

with mlflow.start_run(run_name="linear-regression"):

    model = LinearRegression()
    model.fit(X_train, y_train)

    preds = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, preds))

    mlflow.log_param("model", "LinearRegression")
    mlflow.log_metric("rmse", rmse)

    mlflow.sklearn.log_model(model, "model")

    print("RMSE:", rmse)

2026/03/29 15:01:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/29 15:01:48 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


RMSE: 0.1321390563817621
🏃 View run linear-regression at: https://dagshub.com/slosa23/ML-Assignment1.mlflow/#/experiments/0/runs/94a16a89ee2642a282d5f4e8590d86ce
🧪 View experiment at: https://dagshub.com/slosa23/ML-Assignment1.mlflow/#/experiments/0


*Model Training - Random Forest*

In [70]:
from sklearn.ensemble import RandomForestRegressor

with mlflow.start_run(run_name="random-forest"):

    model = RandomForestRegressor(n_estimators=200, max_depth=10)
    model.fit(X_train, y_train)

    preds = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, preds))

    mlflow.log_param("model", "RandomForest")
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("max_depth", 10)
    mlflow.log_metric("rmse", rmse)

    mlflow.sklearn.log_model(model, "model")

    print("RMSE:", rmse)

2026/03/29 15:02:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/29 15:02:05 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


RMSE: 0.14681523764495008
🏃 View run random-forest at: https://dagshub.com/slosa23/ML-Assignment1.mlflow/#/experiments/0/runs/96ccd967a9044bf6a42a94370852fc62
🧪 View experiment at: https://dagshub.com/slosa23/ML-Assignment1.mlflow/#/experiments/0


*Potential Overfit*

In [71]:
with mlflow.start_run(run_name="potential-overfit"):

    model = RandomForestRegressor(n_estimators=500, max_depth=None)
    model.fit(X_train, y_train)

    preds = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, preds))

    mlflow.log_param("model", "RF_overfit")
    mlflow.log_metric("rmse", rmse)

    print("RMSE:", rmse)

RMSE: 0.14663303374792416
🏃 View run potential-overfit at: https://dagshub.com/slosa23/ML-Assignment1.mlflow/#/experiments/0/runs/88a4f7d13d4847879e66f2e368c708a2
🧪 View experiment at: https://dagshub.com/slosa23/ML-Assignment1.mlflow/#/experiments/0


*Potential Underfit*

In [ ]:
with mlflow.start_run(run_name="potential_underfit"):

    model = RandomForestRegressor(n_estimators=10, max_depth=2)
    model.fit(X_train, y_train)

    preds = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, preds))

    mlflow.log_param("model", "RF_underfit")
    mlflow.log_metric("rmse", rmse)

    print("RMSE:", rmse)